In [2]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [15]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
 C    0.000    0.784    0.000
 C    0.679   -0.392    0.000
 C   -0.679   -0.392    0.000
 H    0.000    1.864    0.000
 H    1.614   -0.932    0.000
 H   -1.614   -0.932    0.000'''
molecule = 'C3H3_cat'
spin = 0
charge =  1


In [16]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [17]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [18]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C3H3_cat.txt
orbital  energy   occ
0     -11.615030451199418    2.0
1     -11.613587273382608    2.0
2     -11.613585029476248    2.0
3     -1.5367428690863878    2.0
4     -1.0666438151784374    2.0
5     -1.0665972794565373    2.0
6     -0.9850653711092847    2.0
7     -0.7971214870013772    2.0
8     -0.7970500719082867    2.0
9     -0.7787610198181087    2.0
10     -0.0514508365095555    0.0
11     -0.05141742523399533    0.0
12     0.23468904162629406    0.0
13     0.2730879091892549    0.0
14     0.2732358034477321    0.0
15     0.3241056593321427    0.0
16     0.7222951611465513    0.0
17     0.7223017953828731    0.0
Homo_index: 9, Lumo_index: 10


WARN: System has nonzero charge 1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]


In [19]:
n_elec= 2
n_orb = 3 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([10,11,12])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-114.76371182928098),
 np.float64(-2.0425144889578064),
 FCIvector([[-9.85485500e-01,  0.00000000e+00,  3.84464165e-06],
            [ 0.00000000e+00,  1.20041698e-01,  0.00000000e+00],
            [ 3.84464165e-06,  0.00000000e+00,  1.20034661e-01]]),
 NPArrayWithTag([[ 5.76380438e-01,  8.11654769e-01,  0.00000000e+00,
                  -1.42404505e-01,  0.00000000e+00, -1.60561609e-01,
                  -4.02167924e-02,  0.00000000e+00, -1.53886653e-02,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                   0.00000000e+00, -1.87340653e-01,  0.00000000e+00,
                   6.32981753e-02,  0.00000000e+00,  6.88950667e-02],
                 [ 7.80820821e-03,  2.15169988e-02,  0.00000000e+00,
                   3.20900387e-01,  0.00000000e+00,  5.42584046e-01,
                   1.96162536e-01,  0.00000000e+00,  1.06521275e-01,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                   0.00000000e+00,  1.293079

In [20]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0]          [0]          -0.985485500104995
[0]          [2]          0.000003844641648
[1]          [1]          0.120041698272006
[2]          [0]          0.000003844641648
[2]          [2]          0.120034660532772
